# 두 겨울 모델 제출 검토

포함 자료만으로 학습 범위, 실제 추론, 개발 지표를 검산합니다. 기본 실행은 재학습하지 않습니다. 보호 3곳의 정답은 사용하지 않습니다. 두 겨울은 반복 개발 자료이며 독립 최종평가가 아닙니다.

In [ ]:
from pathlib import Path
import json, sys
import joblib, numpy as np, pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'training_plan.json').exists())
sys.path.insert(0, str(ROOT))
from src.data_source import sha256_file as sha
from src.two_winter import validate
from src.runtime_metrics import runtime_daily as daily
from src.temperature_transfer import regression_report
from src.submission_runtime import infer
from src.submission_io import read_weather
from src.offline_runtime import deny_network
read = lambda p: json.loads((ROOT/p).read_text('utf-8'))
plan = read('training_plan.json')
assert plan['second_winter_training'] and not plan['protected_outcomes_used']
plan['periods']

## 1. 학습 범위와 모델 무결성

비보호 23곳의 두 겨울. 최종 적합 793,149행. 현재 온도 모델은 MSE·MAE 고정 평균이며 별도 분류기를 채택하지 않았습니다.

In [ ]:
training = validate(pd.read_parquet(ROOT/'data/training_weather_temperature.parquet'))
assert set(training.station) == set(plan['allowed_stations'])
assert not set(training.station) & {'12410','33411','23402'}
assert len(training) == 793149
assert sha(ROOT/'data/training_weather_temperature.parquet') == plan['training_data_sha256']
receipts = read('model/fit_receipts.json')
for receipt in receipts:
    assert sha(ROOT/'model'/(receipt['name']+'.joblib')) == receipt['model_sha256']
pd.DataFrame(receipts)[['name','training_rows','fit_seconds']]

## 2. 외부 통신 없는 실제 추론

학습 범위 내 예제 하루 144행을 실행합니다. 이 일치 검사는 성능평가가 아닙니다.

In [ ]:
models = [joblib.load(ROOT/'model'/(r['name']+'.joblib')) for r in receipts]
with deny_network() as attempts:
    prediction = infer(read_weather(ROOT/'data/weather_example.csv'), models)
assert not attempts
expected = pd.read_csv(ROOT/'data/example_predictions.csv', float_precision='round_trip', parse_dates=['time'], dtype={'station':str,'predicted_ice':'Int64'})
pd.testing.assert_frame_equal(prediction, expected, check_exact=True)
prediction.head()

## 3. 공간 제외 개발 지표 검산

아래는 최종 모델의 학습 내 오차가 아닙니다. 지역을 두 겨울 모두 제외한 별도 개발 예측입니다. 기존 시간 보류 자료를 개발로 편입했으므로 새 후보의 독립평가라고 해석하지 않습니다.

In [ ]:
frame = validate(pd.read_parquet(ROOT/'research/two_winter/audited_predictions.parquet'))
audit = read('research/two_winter/audit.json')
regression = regression_report(frame, frame.temperature_both)
summary = read('research/two_winter/summary.json')
np.testing.assert_allclose(regression['pooled']['mae'], summary['regression']['all']['both']['pooled']['mae'], atol=1e-12, rtol=0)
classification, stations, days = daily(frame, frame.physical_both)
np.testing.assert_allclose(classification['f1_fixed_population'], audit['physical']['both']['all']['f1_fixed_population'], atol=1e-12, rtol=0)
print('온도 MAE:', regression['pooled']['mae'])
print('일별 지점 F1:', classification['f1_fixed_population'])
pd.read_csv(ROOT/'research/two_winter/temperature_regions.csv')

## 4. 제출 CSV와 원래 예측의 일치

보호 예측 103,680행 중 입력 부족 18행은 NA를 유지합니다. 구간별 10분 자료는 그룹 CSV와 위치 인덱스를 연결합니다. 독립 검증 표본 수는 구간 수가 아닙니다.

In [ ]:
receipt = read('exports/manifest.json')
assert receipt['development_scope'] == 'two_winter'
for table in receipt['tables']:
    assert sha(ROOT/'exports'/table['file']) == table['sha256']
protected = pd.read_csv(ROOT/'exports/protected_predictions_10min.csv', dtype={'station':str})
assert len(protected)==103680 and protected.predicted_ice.isna().sum()==18
pd.DataFrame(receipt['tables'])[['file','rows','bytes']]

## 재학습과 전체 연구의 경계

`python -X utf8 code/train.py --output validation/my_refit`으로 포함 학습 자료에서 최종 두 모델을 새 폴더에 재학습할 수 있습니다. 기존 폴더는 덮어쓰지 않습니다. 전체 탐색 실험은 원 비공개 작업 공간의 데이터 경로를 사용하므로 이 패키지만으로 모든 과거 실험을 새로 실행한다고 주장하지 않습니다. 최신 보고서는 `reports/current/current_report.pdf`, 지도는 `figures/`, 데모는 `demo/index.html`입니다.